# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [41]:
import pandas as pd
import numpy as np

march = pd.read_parquet(
    r"C:\Users\ahmad\Documents\GitHub\internship-warehouse\fact_content_daily_performance\month=2026-03"
)

In [42]:
march["ctr"] = (
    march["gsc_clicks"] /
    march["gsc_impressions"].replace(0, np.nan)
)


In [43]:
march["ctr_bucket"] = pd.cut(
    march["ctr"],
    bins=[0,0.01,0.03,0.05,1],
    labels=["0-1%","1-3%","3-5%",">5%"]
)

In [44]:
signal1 = (
    march.groupby("ctr_bucket")
         .agg(
             avg_position=("gsc_avg_position","mean"),
             avg_clicks=("gsc_clicks","mean"),
             n=("gsc_clicks","size")
         )
)

print(signal1)

            avg_position  avg_clicks       n
ctr_bucket                                  
0-1%            9.165533    1.937428  221217
1-3%            7.941102    2.231002  132328
3-5%            8.362170    1.743911   29724
>5%             9.473687    1.330376   34712


For Signal 1 Verdict: MIXED

The relationship between CTR buckets and average clicks/average position is not consistent across the buckets. The 1–3% CTR bucket performs better than both lower and higher CTR buckets. This suggests that CTR alone is not a reliable signal for identifying refresh opportunities in this dataset. Therefore, CTR will not be the primary signal in the baseline rule.

In [45]:
march["position_bucket"] = pd.cut(
    march["gsc_avg_position"],
    bins=[0, 5, 10, 20, 50, float("inf")],
    labels=["1-5", "6-10", "11-20", "21-50", "50+"]
)

signal2 = (
    march.groupby("position_bucket", observed=True)
         .agg(
             avg_clicks=("gsc_clicks", "mean"),
             avg_impressions=("gsc_impressions", "mean"),
             n=("gsc_clicks", "size")
         )
)

print(signal2)

                 avg_clicks  avg_impressions        n
position_bucket                                      
1-5                0.404834       110.400693  1099936
6-10               0.222543        76.009757   920359
11-20              0.178053        56.596118   519223
21-50              0.121283        88.590989   631491
50+                0.005450        12.527727   276863


For Signal 2 Verdict: CONFIRMED

Reason:

Pages with better average search positions generally receive more clicks. Although impressions are not perfectly monotonic across all buckets, click performance steadily decreases as average position worsens. This supports using average position as one of the baseline scoring signals.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [46]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print(march[features].describe())

page_scores = (
    march.groupby("content_hash_id", as_index=False)
    .agg(
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        total_organic_sessions=("sessions_organic", "sum")
    )
)

page_scores = page_scores[
    page_scores["total_impressions"] > 0
].copy()

print(page_scores.shape)

print(page_scores.head())
print(page_scores.shape)

       gsc_impressions    gsc_clicks  gsc_avg_position
count     9.841378e+06  9.841378e+06      3.611061e+06
mean      2.851812e+01  8.350782e-02      1.582665e+01
std       1.559266e+02  7.814341e-01      1.985603e+01
min       0.000000e+00  0.000000e+00      0.000000e+00
25%       0.000000e+00  0.000000e+00      3.742120e+00
50%       0.000000e+00  0.000000e+00      7.500000e+00
75%       6.000000e+00  0.000000e+00      2.020000e+01
max       4.008400e+04  2.740000e+02      4.980000e+02
(176738, 5)
             content_hash_id  total_impressions  total_clicks  avg_position  \
0   content_000005d4ced12088                 86             0     72.854861   
2   content_00007bd2985b77c3                 47             0      5.269565   
6   content_0000cd28fbda69f3                 29             0      4.251282   
8   content_0000d495bfbfb4a8                 15             0      3.333333   
11  content_00014efc121d911d                116             1      4.964683   

    total_organic_

In [47]:
print(page_scores[[
    "total_impressions",
    "total_clicks",
    "avg_position"
]].describe())

       total_impressions   total_clicks   avg_position
count      176738.000000  176738.000000  176738.000000
mean         1587.986675       4.650002      15.999277
std          5431.337724      26.722649      17.686260
min             1.000000       0.000000       0.000000
25%            20.000000       0.000000       5.001970
50%           173.000000       0.000000       8.505296
75%          1039.000000       2.000000      20.369190
max        617124.000000    5668.000000     309.000000


In [48]:
page_scores["score"] = 0

# High visibility
page_scores.loc[
    page_scores["total_impressions"] > 1039,
    "score"
] += 2

# Poor ranking
page_scores.loc[
    page_scores["avg_position"] > 20,
    "score"
] += 2

# Very few clicks
page_scores.loc[
    page_scores["total_clicks"] <= 2,
    "score"
] += 1

print(page_scores["score"].value_counts().sort_index())

score
0     6135
1    89206
2    28300
3    45365
4     4253
5     3479
Name: count, dtype: int64


In [49]:
def reason_code(row):

    if row["avg_position"] > 20:
        return "LOW_POSITION"

    elif row["total_impressions"] > 1039:
        return "HIGH_VISIBILITY"

    elif row["total_clicks"] <= 2:
        return "LOW_CLICKS"

    else:
        return "MONITOR"


page_scores["reason_code"] = page_scores.apply(reason_code, axis=1)

In [50]:
print(page_scores[["score", "reason_code"]].head(10))

    score      reason_code
0       3     LOW_POSITION
2       1       LOW_CLICKS
6       1       LOW_CLICKS
8       1       LOW_CLICKS
11      1       LOW_CLICKS
12      2  HIGH_VISIBILITY
17      1       LOW_CLICKS
19      3     LOW_POSITION
22      0          MONITOR
23      1       LOW_CLICKS


In [51]:
def action(score):

    if score >= 4:
        return "Refresh"

    elif score >= 2:
        return "Review"

    else:
        return "Monitor"


page_scores["action"] = page_scores["score"].apply(action)
print(page_scores[[
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]].head(10))

             content_hash_id  score      reason_code   action
0   content_000005d4ced12088      3     LOW_POSITION   Review
2   content_00007bd2985b77c3      1       LOW_CLICKS  Monitor
6   content_0000cd28fbda69f3      1       LOW_CLICKS  Monitor
8   content_0000d495bfbfb4a8      1       LOW_CLICKS  Monitor
11  content_00014efc121d911d      1       LOW_CLICKS  Monitor
12  content_000184dde41afe75      2  HIGH_VISIBILITY   Review
17  content_0002529572d7880f      1       LOW_CLICKS  Monitor
19  content_0002bd310bf01f15      3     LOW_POSITION   Review
22  content_00032be2df0005ca      0          MONITOR  Monitor
23  content_00033c286cc93446      1       LOW_CLICKS  Monitor


In [52]:
baseline = page_scores.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

baseline.head(10)

import os

os.makedirs("../outputs", exist_ok=True)

baseline.to_csv(
    "../outputs/baseline_action_score.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [53]:
top20 = baseline.head(20)

top20

top20 = top20[[
    "content_hash_id",
    "total_impressions",
    "total_clicks",
    "avg_position",
    "score",
    "reason_code",
    "action"
]]

top20

,content_hash_id,total_impressions,total_clicks,avg_position,score,reason_code,action
0,content_4d0f2d76f82864b4,1488,0,26.285422,5,LOW_POSITION,Refresh
1,content_0b74cdf2bed5ff8c,1310,0,21.421113,5,LOW_POSITION,Refresh
2,content_de5e4d2232691872,1272,0,31.375769,5,LOW_POSITION,Refresh
3,content_de5d5a89f0f74185,6441,2,38.599155,5,LOW_POSITION,Refresh
4,content_0b6a55a3d5e1684c,2235,0,22.957167,5,LOW_POSITION,Refresh
5,content_de798f0833d16117,3070,1,62.240661,5,LOW_POSITION,Refresh
6,content_de9f99ff22823084,16633,1,21.016560,5,LOW_POSITION,Refresh
7,content_deafa0d7f2242942,9864,0,28.429326,5,LOW_POSITION,Refresh
8,content_4d8be266ddc04a5f,1604,0,32.769422,5,LOW_POSITION,Refresh
9,content_4d91140d1970b9df,3246,0,23.358863,5,LOW_POSITION,Refresh


## Top-20 Review

| Content | Action | Reason Code | Confidence | What would make it wrong |
|---------|--------|-------------|------------|--------------------------|
| content_4d0f2d76f82864b4 | Refresh | LOW_POSITION | High | The page may target a highly competitive keyword, so refreshing the content alone may not improve rankings. |
| content_0b74cdf2bed5ff8c | Refresh | LOW_POSITION | High | The low performance may be caused by search intent rather than outdated content. |
| content_de5e4d2232691872 | Refresh | LOW_POSITION | High | The page may already have been updated recently, so the historical data may not reflect its current performance. |
| content_de5d5a89f0f74185 | Refresh | LOW_POSITION | High | Strong competitors or backlink differences may be the main reason for its low ranking. |
| content_0b6a55a3d5e1684c | Refresh | LOW_POSITION | High | Technical SEO issues rather than content quality may be limiting its performance. |
| content_de798f0833d16117 | Refresh | LOW_POSITION | High | The keyword may be extremely competitive, making improvements difficult even after refreshing the page. |
| content_de9f99ff22823084 | Refresh | LOW_POSITION | High | SERP features or user intent may naturally reduce click-through rates. |
| content_deafa0d7f2242942 | Refresh | LOW_POSITION | High | The content may already satisfy user intent, with ranking limited by external factors. |
| content_4d8be266ddc04a5f | Refresh | LOW_POSITION | High | Ranking may depend more on domain authority than page quality. |
| content_4d91140d1970b9df | Refresh | LOW_POSITION | High | The page could already be scheduled for optimization, making this recommendation unnecessary. |
| content_4d98a8ebef58a954 | Refresh | LOW_POSITION | High | The search query may simply be too competitive for meaningful improvement. |
| content_4daf2e9905721747 | Refresh | LOW_POSITION | High | Seasonal search demand could explain the observed performance. |
| content_4db27e4b8d17151d | Refresh | LOW_POSITION | High | Low ranking may be caused by backlink deficiencies rather than content quality. |
| content_4cdeae5c1d0750ef | Refresh | LOW_POSITION | High | The page may target informational queries where low click rates are expected. |
| content_4dccfc323ada11e4 | Refresh | LOW_POSITION | High | Recent improvements may not yet be reflected in the March 2026 data. |
| content_f6ef2355d69535f0 | Refresh | LOW_POSITION | High | External ranking signals may have a greater impact than content changes. |
| content_ddd0f729b12e6791 | Refresh | LOW_POSITION | High | The page may require technical SEO fixes instead of a content refresh. |
| content_de0332567c0725fd | Refresh | LOW_POSITION | High | Search intent may not match the page's content. |
| content_de0e840d0860a69a | Refresh | LOW_POSITION | High | SERP layout changes or rich results may reduce clicks regardless of content quality. |
| content_deab63469b29d9d7 | Refresh | LOW_POSITION | High | Improvements may require stronger site authority or backlinks rather than content updates. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The baseline rule is intentionally simple and may incorrectly recommend some pages for refresh. A page can receive a high score because it has high impressions, low clicks, and a poor average position, even if the real problem is unrelated to the content. For example, the page may target highly competitive keywords, have weak backlinks, suffer from technical SEO issues, or already be in the process of improving. These cases should be reviewed by a human before any action is taken.

## Leakage Check

I confirmed that no future information or label-derived features were used when building the baseline score. The rule only uses historical metrics available at the decision time from the March 2026 dataset:
- Total impressions
- Total clicks
- Average position

No future performance windows, manually created labels, product flags, or outcome variables were included in the score. This keeps the baseline realistic and prevents data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.